# Notebook 01 — Exploratory Data Analysis
## Agentic AI Governance Framework for Multimodal Prompt Injection Detection
> Run this notebook FIRST before any training. Results inform all subsequent phases.

In [ ]:
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from collections import Counter

# ── Seeds for reproducibility ─────────────────────────────────────────────────────
import random
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Path configuration (Colab + Local) ───────────────────────────────────────────
COLAB_BASE = "/content/drive/MyDrive/PAS"
LOCAL_BASE = str(Path(".").resolve().parent)

if os.path.exists(COLAB_BASE):
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    BASE = LOCAL_BASE

DATASET_DIR = os.path.join(BASE, "FINAL_BENCHMARK_DATASET")
CACHE_DIR   = os.path.join(BASE, "EXPERIMENT_CACHE")
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Dataset dir: {DATASET_DIR}")
print(f"Exists: {os.path.exists(DATASET_DIR)}")

# ── Style ───────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150, 'font.family': 'sans-serif',
    'axes.spines.top': False, 'axes.spines.right': False
})
print("Setup complete.")

In [ ]:
# Load parquet splits
print("Loading train/val/test parquets...")
train_df = pd.read_parquet(os.path.join(DATASET_DIR, "train.parquet"))
val_df   = pd.read_parquet(os.path.join(DATASET_DIR, "validation.parquet"))
test_df  = pd.read_parquet(os.path.join(DATASET_DIR, "test.parquet"))

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df["split"] = name

combined = pd.concat([train_df, val_df, test_df], ignore_index=True)
print(f"Total rows: {len(combined):,}")
print(f"Columns ({len(combined.columns)}): {list(combined.columns)}")
combined.head(3)

In [ ]:
# 1. Class balance per split
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, df) in zip(axes, [("train", train_df), ("val", val_df), ("test", test_df)]):
    counts = df["label"].value_counts()
    colors = ["#2ecc71" if l == "benign" else "#e74c3c" for l in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor="white", linewidth=1.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                f"{val:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    total = len(df)
    mal_pct = (df["label"] == "malicious").mean()
    ax.set_title(f"{name.upper()} (n={total:,})\nMalicious: {mal_pct:.1%}",
                 fontsize=12, fontweight="bold")
    ax.set_ylabel("Count")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.suptitle("Class Balance per Split", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CACHE_DIR, "01_class_balance.png"), bbox_inches="tight")
plt.show()
print(f"Overall malicious rate: {(combined['label']=='malicious').mean():.2%}")

In [ ]:
# 2. Attack family distribution
if "attack_family" in combined.columns:
    mal_df = combined[combined["label"] == "malicious"]
    fam_counts = mal_df["attack_family"].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(12, 5))
    palette = sns.color_palette("husl", len(fam_counts))
    bars = ax.barh(fam_counts.index[::-1], fam_counts.values[::-1], color=palette[::-1])
    for bar, val in zip(bars, fam_counts.values[::-1]):
        ax.text(val + 10, bar.get_y() + bar.get_height()/2,
                f"{val:,}", va="center", fontsize=9)
    ax.set_xlabel("Count")
    ax.set_title("Attack Family Distribution (Malicious Samples)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(CACHE_DIR, "01_attack_families.png"), bbox_inches="tight")
    plt.show()
    print("Attack families found:", list(fam_counts.index))
else:
    print("WARNING: 'attack_family' column not found")

In [ ]:
# 3. Token length distribution (critical for RoBERTa 512-token limit)
TEXT_COL = next((c for c in ["text", "ocr_text", "prompt"] if c in combined.columns), None)
if TEXT_COL:
    print(f"Using text column: '{TEXT_COL}'")
    combined["word_count"] = combined[TEXT_COL].fillna("").str.split().str.len()
    # Approximate token count: words * 1.3
    combined["approx_tokens"] = (combined["word_count"] * 1.3).astype(int)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, label, color in zip(axes, ["benign", "malicious"], ["#2ecc71", "#e74c3c"]):
        subset = combined[combined["label"] == label]["approx_tokens"]
        ax.hist(subset, bins=50, color=color, alpha=0.8, edgecolor="white")
        ax.axvline(512, color="black", linestyle="--", linewidth=2, label="512-token limit")
        pct_over = (subset > 512).mean()
        ax.set_title(f"{label.upper()}\n{pct_over:.1%} exceed 512 tokens",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("Approximate Token Count")
        ax.set_ylabel("Count")
        ax.legend()
    plt.suptitle("Token Length Distribution (RoBERTa 512-token limit shown)",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(CACHE_DIR, "01_token_lengths.png"), bbox_inches="tight")
    plt.show()
else:
    print("WARNING: No text column found")

In [ ]:
# 4. Data integrity: split disjointness + image existence
print("=" * 60)
print("DATA INTEGRITY CHECKS")
print("=" * 60)

# 4a. Split disjointness
train_ids = set(train_df["sample_id"])
val_ids   = set(val_df["sample_id"])
test_ids  = set(test_df["sample_id"])

for a_name, a_ids, b_name, b_ids in [
    ("train", train_ids, "val", val_ids),
    ("train", train_ids, "test", test_ids),
    ("val",   val_ids,  "test", test_ids),
]:
    overlap = len(a_ids & b_ids)
    status = "\u2705" if overlap == 0 else "\u274c LEAKAGE!"
    print(f"  {status} {a_name} \u2229 {b_name}: {overlap} overlapping samples")

# 4b. Image existence
if "image_path" in combined.columns:
    img_rows = combined[combined["image_path"].notna()].copy()
    img_rows["full_path"] = img_rows["image_path"].apply(
        lambda p: os.path.join(DATASET_DIR, "IMAGES", os.path.basename(str(p)))
        if p and not os.path.isabs(str(p)) else str(p)
    )
    exists = img_rows["full_path"].apply(os.path.exists)
    print(f"\n  Image-backed samples: {len(img_rows):,}")
    print(f"  Files found on disk:  {exists.sum():,} ({exists.mean():.1%})")
    print(f"  Files missing:        {(~exists).sum():,}")
    missing = img_rows[~exists]["full_path"].head(3).tolist()
    if missing:
        print(f"  Example missing paths: {missing}")
else:
    print("  No 'image_path' column found \u2014 skip image check")

# 4c. runtime_ocr column
if "runtime_ocr" in combined.columns:
    print(f"\n  runtime_ocr dtype: {combined['runtime_ocr'].dtype}")
    print(f"  Value counts:\n{combined['runtime_ocr'].value_counts().head()}")
    print("  -> Text column likely IS OCR output" if combined["runtime_ocr"].astype(str).str.lower().isin(["true","1"]).mean() > 0.5
          else "  -> TEXT COLUMN MAY BE SYNTHETIC (NOT OCR) \u2014 verify before RoBERTa training!")

print("\nDone. Save these findings before proceeding to Notebook 02.")

In [ ]:
# 5. Save dataset card (for paper + reproducibility)
card = {
    "dataset_version": "v1.0",
    "total_samples": int(len(combined)),
    "train_samples": int(len(train_df)),
    "val_samples":   int(len(val_df)),
    "test_samples":  int(len(test_df)),
    "label_distribution": combined["label"].value_counts().to_dict(),
    "image_backed_samples": int(combined["image_path"].notna().sum()) if "image_path" in combined.columns else 0,
    "columns": list(combined.columns),
    "random_seed": RANDOM_SEED,
    "attack_families": combined["attack_family"].unique().tolist() if "attack_family" in combined.columns else [],
}
card_path = os.path.join(CACHE_DIR, "dataset_card.json")
with open(card_path, "w") as f:
    json.dump(card, f, indent=2, default=str)
print(f"Dataset card saved to: {card_path}")
print(json.dumps(card, indent=2, default=str))